# LightGBM + TabTransformer 블렌딩 효과 비교

`1차_lgbm_oof_only.ipynb`와 `1차_tt_oof_only.ipynb`를 각각 먼저 실행해서
`blend_cache/` 폴더에 결과가 저장되어 있어야 합니다.

이 노트북은 가벼운 numpy 연산만 하므로 라이브러리 충돌이나 속도 문제가 전혀 없습니다.

In [1]:
import numpy as np
from sklearn.metrics import roc_auc_score

CACHE_DIR = "blend_cache"

oof_lgbm = np.load(f"{CACHE_DIR}/oof_lgbm.npy")
oof_tt = np.load(f"{CACHE_DIR}/oof_tt.npy")
y = np.load(f"{CACHE_DIR}/oof_y.npy")

print(f"불러오기 완료: LightGBM {oof_lgbm.shape}, TabTransformer {oof_tt.shape}, y {y.shape}")

불러오기 완료: LightGBM (256351,), TabTransformer (256351,), y (256351,)


## 블렌딩 효과 확인

In [2]:
score_lgbm = roc_auc_score(y, oof_lgbm)
score_tt = roc_auc_score(y, oof_tt)
corr = np.corrcoef(oof_lgbm, oof_tt)[0, 1]

print(f"LightGBM 5-Fold OOF: {score_lgbm:.5f}")
print(f"TabTransformer 5-Fold OOF: {score_tt:.5f}")
print(f"OOF 예측 상관계수: {corr:.4f}")
print()
print("=== 가중평균 그리드서치 ===")
best_w, best_score = 1.0, score_lgbm
for w in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]:
    blend = w * oof_lgbm + (1 - w) * oof_tt
    blend_score = roc_auc_score(y, blend)
    marker = " <-- 현재 최고" if blend_score > best_score else ""
    if blend_score > best_score:
        best_w, best_score = w, blend_score
    print(f"LightGBM {w:.2f} + TabTransformer {1-w:.2f}: {blend_score:.5f}{marker}")

print(f"\n최적 가중치: LightGBM {best_w:.2f} + TabTransformer {1-best_w:.2f} (점수: {best_score:.5f})")
print(f"LightGBM 단독 대비 개선: {best_score - score_lgbm:+.5f}")

LightGBM 5-Fold OOF: 0.73925
TabTransformer 5-Fold OOF: 0.73523
OOF 예측 상관계수: 0.9686

=== 가중평균 그리드서치 ===
LightGBM 0.50 + TabTransformer 0.50: 0.73923
LightGBM 0.60 + TabTransformer 0.40: 0.73955 <-- 현재 최고
LightGBM 0.70 + TabTransformer 0.30: 0.73971 <-- 현재 최고
LightGBM 0.75 + TabTransformer 0.25: 0.73973 <-- 현재 최고
LightGBM 0.80 + TabTransformer 0.20: 0.73971
LightGBM 0.85 + TabTransformer 0.15: 0.73965
LightGBM 0.90 + TabTransformer 0.10: 0.73955
LightGBM 0.95 + TabTransformer 0.05: 0.73942
LightGBM 1.00 + TabTransformer 0.00: 0.73925

최적 가중치: LightGBM 0.75 + TabTransformer 0.25 (점수: 0.73973)
LightGBM 단독 대비 개선: +0.00047


## 결론

개선이 노이즈 수준(대략 std 0.002 미만)이라면 블렌딩은 채택할 가치가 낮습니다.
의미 있는 개선(+0.001 이상, 가능하면 다른 random_state로도 재확인)이라면,
이 가중치를 사용해서 test 예측도 같은 방식으로 블렌딩하여 제출 파일을
만들 수 있습니다 (해당 코드는 결과가 긍정적일 때 추가로 작성).